# CS102 Workshop 1: อ่าน Text File ด้วย `while`

## Student Info 
1. Student ID:1690700933
2. Name : jirayos tippason
3. Section :127G

## หัวข้อ: Read file ทีละบรรทัดด้วย `while loop`

**สถานการณ์:** โปรแกรมต้องอ่านคะแนนนักศึกษาจากไฟล์ข้อความ แล้วเลือกเฉพาะข้อมูลที่ต้องการ

> ก่อนเริ่ม: นักศึกษาเคยเรียน `tuple`, `list`, `for loop` และ `if-else` แล้ว

**ผลลัพธ์การเรียนรู้**
1. อธิบายได้ว่า `readline()` คืนค่าอะไร และรู้ได้อย่างไรว่าถึงท้ายไฟล์ (EOF)
2. ใช้ `while` อ่านไฟล์ทีละบรรทัดได้อย่างปลอดภัย
3. ใช้ `strip()`, `split()`, `if-else` และ `list.append()` ร่วมกับข้อมูลจากไฟล์ได้
4. เขียนโปรแกรมกรองและสรุปข้อมูลอย่างง่ายจาก text file ได้

**คำแนะนำ:** รัน cell ตามลำดับจากบนลงล่าง ไฟล์ตัวอย่างจะถูกสร้างในโฟลเดอร์เดียวกับ Notebook นี้


## 1. สร้างข้อมูลตัวอย่าง

ในโลกจริงไฟล์อาจได้รับจากอาจารย์หรือระบบอื่น ใน workshop นี้เราสร้างไฟล์ `scores.txt` ขึ้นมาก่อน

รูปแบบข้อมูลที่ถูกต้องคือ `รหัสนักศึกษา,ชื่อ,คะแนน` หนึ่งระเบียนต่อหนึ่งบรรทัด
โดยตั้งใจใส่บรรทัดว่างและบรรทัดที่ผิดรูปแบบไว้ เพื่อฝึกตรวจสอบข้อมูลด้วย `if-else`


In [2]:
from pathlib import Path

FILE_PATH = Path("scores.txt")
sample_data = (
    "1670712345 Chonnawee Phiphatkul 95\n"
    "1670712346 Shalisa Bunyasarn 75\n"
    "1670754321 Nuanjan Yongjaiyut 60\n"
    "1670712348 Yada Nimitwanitch 100\n"
    "invalid row\n"
)

FILE_PATH.write_text(sample_data, encoding="utf-8")
print(f"สร้างไฟล์แล้ว: {FILE_PATH.resolve()}")


สร้างไฟล์แล้ว: C:\jirayos_01\w10\scores.txt


In [3]:
# ตรวจดูข้อความดิบในไฟล์ก่อนเริ่มอ่านทีละบรรทัด
print(FILE_PATH.read_text(encoding="utf-8"))


1670712345 Chonnawee Phiphatkul 95
1670712346 Shalisa Bunyasarn 75
1670754321 Nuanjan Yongjaiyut 60
1670712348 Yada Nimitwanitch 100
invalid row



## 2. แนวคิดสำคัญ: `readline()` และ EOF

เมื่อเปิดไฟล์ Python มี **ตำแหน่งการอ่าน (file pointer)** อยู่ที่ต้นไฟล์

```python
line = file.readline()
```

- ถ้ายังมีข้อมูล `line` จะเป็นข้อความ **รวม `\n` ท้ายบรรทัด** (ถ้ามี)
- เมื่ออ่านถึงท้ายไฟล์ `line` จะเป็นสตริงว่าง `""` ซึ่งเรียกว่า **EOF — End Of File**
- ดังนั้นจึงใช้ `while True` แล้ว `break` เมื่อ `line == ""`

ใช้ `with open(...) as file:` เสมอ เพราะ Python จะปิดไฟล์ให้อัตโนมัติ แม้เกิดข้อผิดพลาด


In [5]:
# รูปแบบมาตรฐาน: อ่านทีละบรรทัดด้วย while
with open(FILE_PATH, "r", encoding="utf-8") as file:
    line_number = 1

    while True:
        line = file.readline()

        if line == "":          # ถึง EOF แล้ว
            break

        print(f"line {line_number}: {line!r}")
        line_number += 1


line 1: '1670712345 Chonnawee Phiphatkul 95\n'
line 2: '1670712346 Shalisa Bunyasarn 75\n'
line 3: '1670754321 Nuanjan Yongjaiyut 60\n'
line 4: '1670712348 Yada Nimitwanitch 100\n'
line 5: 'invalid row\n'


สังเกตว่า `!r` ใน `print` แสดงเครื่องหมายพิเศษ เช่น `\n` เพื่อให้เห็นว่า `readline()` ยังมี newline ติดมาด้วย

เมื่อต้องการนำข้อความไปใช้งาน ให้เรียก `line.strip()`:

- ลบช่องว่างและ `\n` ที่หัว/ท้ายข้อความ
- ถ้าบรรทัดว่าง `line.strip() == ""`

> `for line in file:` ก็อ่านไฟล์ได้เช่นกัน แต่ workshop นี้ฝึกควบคุมการวนซ้ำและจุดสิ้นสุดด้วย `while`


## 3. ตัวอย่าง: เก็บเฉพาะระเบียนที่ถูกต้องลง list

โปรแกรมด้านล่างจะข้ามบรรทัดว่างและบรรทัดที่ไม่ได้มีข้อมูล 3 ส่วน จากนั้นเก็บข้อมูลที่ผ่านการตรวจสอบไว้ใน list


In [6]:
valid_records = []
skipped_lines = 0

with open(FILE_PATH, "r", encoding="utf-8") as file:
    while True:
        line = file.readline()

        if line == "":
            break

        clean_line = line.strip()

        if clean_line == "":
            skipped_lines += 1
        else:
            parts = clean_line.split(" ")

            if len(parts) == 4:
                valid_records.append(parts)
            else:
                skipped_lines += 1

print("ข้อมูลที่ใช้ได้:", valid_records)
print("จำนวนบรรทัดที่ข้าม:", skipped_lines)


ข้อมูลที่ใช้ได้: [['1670712345', 'Chonnawee', 'Phiphatkul', '95'], ['1670712346', 'Shalisa', 'Bunyasarn', '75'], ['1670754321', 'Nuanjan', 'Yongjaiyut', '60'], ['1670712348', 'Yada', 'Nimitwanitch', '100']]
จำนวนบรรทัดที่ข้าม: 1


## แบบฝึกหัด 1: แสดงข้อมูลแบบมีหมายเลข

เขียนโปรแกรมอ่าน `scores.txt` ด้วย `while` แล้วแสดง **เฉพาะบรรทัดที่ไม่ว่าง** ในรูปแบบนี้


**เงื่อนไข**
- ต้องใช้ `readline()` ภายใน `while`
- ต้องหยุดเมื่อถึง EOF
- ห้ามเพิ่มหมายเลขให้บรรทัดว่าง

เขียนคำตอบใน cell ถัดไป แล้วรันเพื่อตรวจผลลัพธ์


In [10]:
# TODO: เขียนคำตอบแบบฝึกหัด 1 ที่นี่
# Hint: สร้างตัวแปร number = 1 ก่อนเปิดไฟล์

number = 1

with open("scores.txt", "r") as file:
    while True:
        line = file.readline()

        clean_line = line.strip()

        if clean_line == "":
            break
        if clean_line == "invalid row":
            continue
        number += 1
        print(f"line {number}: {clean_line!r}")



line 2: '1670712345 Chonnawee Phiphatkul 95'
line 3: '1670712346 Shalisa Bunyasarn 75'
line 4: '1670754321 Nuanjan Yongjaiyut 60'
line 5: '1670712348 Yada Nimitwanitch 100'


## แบบฝึกหัด 2: รายชื่อผู้ผ่านเกณฑ์

อ่าน `scores.txt` แล้วสร้าง list ชื่อ `passed_names` สำหรับนักศึกษาที่มีคะแนน **ตั้งแต่ 80 ขึ้นไป**

**กติกา**
1. ข้ามบรรทัดว่าง
2. ข้ามบรรทัดที่มีส่วนข้อมูลไม่ครบ 3 ส่วน
3. แปลงคะแนนด้วย `int()` แล้วใช้ `if-else` ตัดสินผล
4. ใช้ `append()` เพิ่มเฉพาะชื่อผู้ผ่านเกณฑ์

ผลลัพธ์ที่คาดหวัง: `['Ada', 'Chai']`


In [17]:
# TODO: เขียนคำตอบแบบฝึกหัด 2 ที่นี่



passed_names = []

with open("scores.txt", "r") as file:
    while True:
        line = file.readline()

        clean_line = line.strip()

        if clean_line == "":
            break
        parts = clean_line.split(" ")

        if len(parts) != 4:
            continue

        score = int(parts[3])
        name = parts[1]

        if score >= 80:
            passed_names.append(name)
        
    print(passed_names)



['Chonnawee', 'Yada']


## โจทย์ท้าทาย: สรุปคะแนน

ใช้ `while` อ่านเฉพาะระเบียนที่ถูกต้อง แล้วหาค่า

- จำนวนผู้มีคะแนนตั้งแต่ 80 ขึ้นไป
- จำนวนผู้มีคะแนนต่ำกว่า 80
- คะแนนรวม และคะแนนเฉลี่ย (กรณีมีข้อมูลอย่างน้อยหนึ่งระเบียน)

ลองใช้ตัวแปรสะสม เช่น `total_score`, `record_count`, `pass_count` และ `fail_count`
ห้ามใช้ `readlines()` และห้ามใช้ `for`


In [24]:
# TODO: เขียนคำตอบโจทย์ท้าทายที่นี่

pass_score = []
fall_score = []
total_score = 0
total_student = 0
with open("scores.txt", "r") as file:
    while True:
        line = file.readline()

        clean_line = line.strip()

        if clean_line == "":
            break
        parts = clean_line.split(" ")

        if len(parts) != 4:
            continue
        
        score = int(parts[3])
        total_student += 1
        total_score += score
        if score >= 80:
            pass_score.append(score)

        else:
            fall_score.append(score)

print("จำนวนคะแนนสอบผ่าน:", len(pass_score))
print("จำนวนคะแนนสอบตก:", len(fall_score))
print("คะแนนรวม", total_score)
print("คะแนนเฉลี่ย:", total_score / total_student)

จำนวนคะแนนสอบผ่าน: 2
จำนวนคะแนนสอบตก: 2
คะแนนรวม 330
คะแนนเฉลี่ย: 82.5


## สรุปและตรวจสอบตนเอง

ตอบคำถามต่อไปนี้ก่อนจบ workshop

1. เพราะเหตุใดจึงไม่ควรเขียน `while True` โดยไม่มี `break` สำหรับการอ่านไฟล์?
2. ค่าใดที่ `readline()` ส่งกลับเมื่อถึง EOF?
3. เหตุใดจึงเรียก `strip()` ก่อน `split(",")`?
4. ถ้าต้องการอ่านจากต้นไฟล์ใหม่ ต้อง `open` ไฟล์ใหม่ หรือใช้ `file.seek(0)`

**รูปแบบจำเป็นที่ควรจำ**

```python
with open("ชื่อไฟล์.txt", "r", encoding="utf-8") as file:
    while True:
        line = file.readline()
        if line == "":
            break
        # ประมวลผล line ตรงนี้
```
